# 12 · H4′: глобальный срез против условного

**Зачем** (решение Павла 2026-10-31, `docs/HYPOTHESES.md` H4′). H4 утверждает: вмешательство в активации выигрывает
у дообучения на **условных** изменениях и проигрывает на **глобальных**. Для классического steering на условном срезе
это уже не так (LoRA сильно лучше CAA). Проверяем ту же идею для методов проекта — аффинных поправок в точке вызова
без градиента (`linmap`, `linmap_prompt`) — против LoRA.

**Глобальный срез** (генератор `g1`, `docs/METRICS.md` «Глобальный срез»). Тот же магазин и те же ответы `check_stock`
в двух ветках (подходит / не подходит), но правильное действие одно: сначала спросить пользователя (`ask_user`).
Правило в промпте не написано — его учат из данных. Контрольные пары: пользователь уже ответил, верное действие
зависит от ответа, спросить снова — ошибка. Принятие правила = столбец `sensitivity`, сохранность = `specificity`.

Учитель `linmap_prompt` здесь — правило словами в system (`policy_prompt`). CAA (`steering`) и `linmap_mix` — справочно, вне H4′.
Условный срез — прогон ноутбука 11 (тот же набор методов, то же правило отбора). Перенос — свежий seed 606.
Результат **предварительный** (test не открыт).


In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # Mac: неподдержанные операции -> CPU

import pandas as pd
from IPython.display import Markdown, display
pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)

from steering_tool_feedback.config import ExperimentConfig
from steering_tool_feedback.pipeline import Experiment
from steering_tool_feedback import journal, viz

In [2]:
# Правило отбора — то же, что для условного среза (запись в журнале до этого прогона).
SELECT = "val+loto"
cfg = ExperimentConfig.make(
    name="global-slice-qwen3-1.7b", hypothesis="explore", notebook="notebooks/12_global_slice.ipynb",
    notes="H4′: глобальный срез g1 (правило ask_user) — linmap, linmap_prompt (учитель policy_prompt), LoRA; linmap_mix и CAA справочно.",
    overrides={
        "model.name": "Qwen/Qwen3-1.7B",
        "data.n_tasks": 160,
        "data.generator": "g1",
        "data.stratify_pair_type": True,
        "eval.methods": ["baseline", "policy_prompt", "steering", "lora", "linmap", "linmap_prompt", "linmap_mix"],
        "steer.alphas": [0.25, 0.5, 1.0, 2.0, 4.0],
        "act.select": SELECT,
        "act.teacher": "policy_prompt",
        "act.ranks": [8],
        "act.alphas": [1.0, 1.5, 2.0, 3.0, 4.0],
        "episode.enabled": False,
        "transfer.seed": 606,
        "transfer.episode_methods": [],
    },
)
exp = Experiment(cfg)
exp.run("fit")
exp.results_table("val")


/Users/admin/Desktop/steering-tool-feedback/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[21:26:22] run 20261101-212619_global-slice-qwen3-1.7b_0a9f9b  ->  results/runs/20261101-212619_global-slice-qwen3-1.7b_0a9f9b
[21:26:22] data: {'train': 80, 'val': 31, 'test': 49}  types={'global': 112, 'control': 48}  positions={'early': 160}


baseline/train:   0%|          | 0/160 [00:00<?, ?it/s]

[21:26:22] loading model Qwen/Qwen3-1.7B …


W1101 21:26:25.719000 2789 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

... (516 outputs left out)

linmap_mix/val:   0%|          | 0/62 [00:00<?, ?it/s]

linmap_prompt@val/val:   0%|          | 0/62 [00:00<?, ?it/s]

[21:26:47] val: baseline 0.339; policy_prompt 0.919; steering 0.387; linmap 0.484; linmap_prompt 0.919; linmap_mix 0.468; linmap_prompt@val 0.919


,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.339,0.339,0.000,1.0,NaN,,NaN,NaN,None,None,,None,798,10.0,391.9
1,policy_prompt,0.919,0.919,0.773,1.0,0.581,"[+0.419, +0.726]",36.0,0.0,None,None,,None,837,10.0,309.5
2,steering,0.387,0.387,0.045,1.0,0.048,"[+0.000, +0.113]",3.0,0.0,None,None,,None,798,10.0,408.2
3,linmap,0.484,0.484,0.136,1.0,0.145,"[+0.065, +0.242]",9.0,0.0,None,None,,None,798,10.0,459.7
4,linmap_prompt,0.919,0.919,0.773,1.0,0.581,"[+0.435, +0.726]",36.0,0.0,None,None,,None,798,10.0,301.9
5,linmap_mix,0.468,0.468,0.136,1.0,0.129,"[+0.048, +0.226]",8.0,0.0,None,None,,None,798,10.0,296.9
6,linmap_prompt@val,0.919,0.919,0.773,1.0,0.581,"[+0.419, +0.726]",36.0,0.0,None,None,,None,798,10.0,300.0


### Подбор на глобальном срезе

In [3]:
f = exp.state["fitted"]
lp = f["linmap_prompt"]
print(f"учитель {lp.get('teacher')}: успех на train {lp.get('teacher_train_success', float('nan')):.3f} "
      f"(база {lp.get('base_train_success', float('nan')):.3f})")
cols = ["layer", "rank", "alpha", "success", "loto_success", "score", "sensitivity", "specificity", "spec_drop"]
for m in ("linmap", "linmap_prompt", "linmap_mix"):
    t = exp.fit_table(m)
    if t.empty:
        print(m, f[m].get("reason")); continue
    r, rv = f[m]["row"], f[m].get("row_val") or f[m]["row"]
    display(Markdown(f"**{m}** — выбрано L{r['layer']}/α{r['alpha']}; по одному val: L{rv['layer']}/α{rv['alpha']}"))
    display(t.sort_values("score" if "score" in t else "success", ascending=False)[[c for c in cols if c in t]].head(6).round(3))


учитель policy_prompt: успех на train 0.884 (база 0.125)


**linmap** — выбрано L16/α1.0; по одному val: L16/α1.0

,layer,rank,alpha,success,loto_success,score,sensitivity,specificity,spec_drop
15,22,8,1.0,0.839,0.887,0.863,0.955,0.222,0.778
2,16,8,2.0,0.758,0.871,0.815,0.636,0.444,0.556
10,20,8,1.0,0.774,0.758,0.766,0.727,0.333,0.667
3,16,8,3.0,0.726,0.798,0.762,0.727,0.111,0.889
25,26,8,1.0,0.758,0.742,0.750,1.000,0.000,1.000
20,24,8,1.0,0.742,0.742,0.742,1.000,0.111,0.889


**linmap_prompt** — выбрано L18/α2.0; по одному val: L20/α1.5

,layer,rank,alpha,success,loto_success,score,sensitivity,specificity,spec_drop
23,24,8,3.0,0.935,0.960,0.948,0.909,0.778,0.222
22,24,8,2.0,0.935,0.952,0.944,0.864,0.889,0.111
27,26,8,2.0,0.935,0.952,0.944,0.864,0.889,0.111
18,22,8,3.0,0.952,0.927,0.940,0.909,0.889,0.111
19,22,8,4.0,0.919,0.935,0.927,0.909,0.778,0.222
7,18,8,2.0,0.919,0.927,0.923,0.773,1.000,0.000


**linmap_mix** — выбрано L16/α1.0; по одному val: L16/α1.0

,layer,rank,alpha,success,loto_success,score,sensitivity,specificity,spec_drop
3,16,8,3.0,0.919,0.927,0.923,0.909,0.667,0.333
15,22,8,1.0,0.903,0.935,0.919,0.864,0.667,0.333
20,24,8,1.0,0.887,0.911,0.899,1.000,0.333,0.667
6,18,8,1.5,0.871,0.903,0.887,0.955,0.333,0.667
4,16,8,4.0,0.887,0.871,0.879,1.000,0.333,0.667
10,20,8,1.0,0.806,0.919,0.863,0.818,0.222,0.778


### Перенос (seed 606): знакомые и новые типы требований

In [4]:
exp.transfer()
for g in ("seen", "unseen"):
    display(Markdown(f"**{g}**"))
    display(exp.results_table(f"transfer:{g}"))


baseline/transfer:seen:   0%|          | 0/100 [00:00<?, ?it/s]

baseline/transfer:seen:  29%|██▉       | 29/100 [00:00<00:00, 281.49it/s]

baseline/transfer:seen:  58%|█████▊    | 58/100 [00:00<00:00, 282.52it/s]

baseline/transfer:seen:  87%|████████▋ | 87/100 [00:00<00:00, 279.43it/s]

policy_prompt/transfer:seen:   0%|          | 0/100 [00:00<?, ?it/s]

policy_prompt/transfer:seen:  28%|██▊       | 28/100 [00:00<00:00, 276.69it/s]

... (22 outputs left out)

**seen**

,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.34,0.34,0.000,1.0,NaN,,NaN,NaN,None,None,,None,800,10.0,306.4
1,policy_prompt,0.99,0.99,0.971,1.0,0.65,"[+0.520, +0.770]",65.0,0.0,None,None,,None,839,10.0,311.9
2,steering,0.38,0.38,0.029,1.0,0.04,"[+0.010, +0.080]",4.0,0.0,None,None,,None,800,10.0,311.8
3,linmap,0.52,0.52,0.200,1.0,0.18,"[+0.090, +0.270]",18.0,0.0,None,None,,None,800,10.0,312.2
4,linmap_prompt,0.98,0.98,0.943,1.0,0.64,"[+0.510, +0.760]",64.0,0.0,None,None,,None,800,10.0,312.6
5,linmap_mix,0.52,0.52,0.171,1.0,0.18,"[+0.100, +0.270]",18.0,0.0,None,None,,None,800,10.0,312.1
6,linmap_prompt@val,0.99,0.99,0.971,1.0,0.65,"[+0.520, +0.770]",65.0,0.0,None,None,,None,800,10.0,312.6


**unseen**

,method,success,pass^1,sensitivity,specificity,diff_vs_base,ci,fixed,broken,episode,ep_diff,ep_ci,stress,tokens,calls,p50_ms
0,baseline,0.41,0.41,0.256,1.000,NaN,,NaN,NaN,None,None,,None,796,10.0,309.0
1,policy_prompt,0.99,0.99,0.977,1.000,0.58,"[+0.450, +0.700]",58.0,0.0,None,None,,None,835,10.0,312.5
2,steering,0.48,0.48,0.302,1.000,0.07,"[+0.030, +0.120]",7.0,0.0,None,None,,None,796,10.0,312.7
3,linmap,0.50,0.50,0.349,1.000,0.09,"[+0.040, +0.150]",9.0,0.0,None,None,,None,796,10.0,313.0
4,linmap_prompt,0.95,0.95,0.884,1.000,0.54,"[+0.410, +0.660]",54.0,0.0,None,None,,None,796,10.0,312.4
5,linmap_mix,0.42,0.42,0.302,1.000,0.01,"[-0.040, +0.060]",4.0,3.0,None,None,,None,796,10.0,311.8
6,linmap_prompt@val,0.96,0.96,0.930,0.857,0.55,"[+0.420, +0.680]",56.0,1.0,None,None,,None,796,10.0,312.4


### H4′: условный срез (ноутбук 11) против глобального

Δ = успех метода − успех LoRA, парно по задачам; I = Δ_усл − Δ_глоб, интервал уровня по Холму (97.5% на два метода).
Знакомые типы — предварительный вывод по H4′; новые типы — разведка.

In [5]:
from steering_tool_feedback import crossslice

COND_RUN = exp.store.results / "runs" / "20261031-160942_loto-select-qwen3-1.7b_bd79ce"   # ноутбук 11
METHODS = {"linmap": ("linmap", "linmap"), "linmap_prompt": ("linmap_prompt", "linmap_prompt")}
h4p = {}
for g in ("seen", "unseen"):
    out = crossslice.compare(COND_RUN, exp.dir, f"transfer:{g}", methods=METHODS,
                             tol=cfg.steer.max_specificity_drop, n_boot=cfg.eval.n_boot)
    h4p[g] = out
    display(Markdown(f"**{g}** (уровень {out['level']:.3f}) — H4′ {out['verdict'].label}: {out['verdict'].reason}"))
    display(pd.DataFrame(out["rows"]).drop(columns=["cond_name", "glob_name"]).round(3))
if any(o.get("ref_missing") for o in h4p.values()):
    # ориентир не выбран: показать, почему его конфигурации не прошли допуск отбора
    display(Markdown("**LoRA на глобальном срезе — строки подбора (val)**"))
    display(exp.fit_table("lora").round(3))
# Разведка (не входит в H4′): linmap_mix — его выигрыш при отборе val+loto замечен после предрегистрации
for g in ("seen", "unseen"):
    out = crossslice.compare(COND_RUN, exp.dir, f"transfer:{g}", methods={"linmap_mix": ("linmap_mix", "linmap_mix")},
                             tol=cfg.steer.max_specificity_drop, n_boot=cfg.eval.n_boot)
    h4p[f"{g}|linmap_mix (разведка)"] = out
    display(Markdown(f"**{g}, linmap_mix — разведка, вне H4′**"))
    display(pd.DataFrame(out["rows"]).drop(columns=["cond_name", "glob_name"]).round(3))
exp._save_json("h4_prime.json", {g: {**{k: v for k, v in o.items() if k != "verdict"},
                                     "verdict": o["verdict"].to_dict()} for g, o in h4p.items()})


**seen** (уровень 0.975) — H4′ · not_run (val): ориентир 'lora' не оценивался на 'transfer:seen' в глобальном срезе: ни одна его конфигурация не прошла допуск отбора; Δ и I не определены, предрегистрация H4′ этот случай не описывает

,method,success_cond,sens_cond,spec_cond,spec_drop_cond,eligible_cond,success_glob,sens_glob,spec_glob,spec_drop_glob,eligible_glob
0,linmap,0.75,0.636,0.529,-0.235,True,0.52,0.200,1.0,0.0,True
1,linmap_prompt,0.73,0.545,0.588,-0.294,True,0.98,0.943,1.0,0.0,True


**unseen** (уровень 0.975) — H4′ · not_run (val): ориентир 'lora' не оценивался на 'transfer:unseen' в глобальном срезе: ни одна его конфигурация не прошла допуск отбора; Δ и I не определены, предрегистрация H4′ этот случай не описывает

,method,success_cond,sens_cond,spec_cond,spec_drop_cond,eligible_cond,success_glob,sens_glob,spec_glob,spec_drop_glob,eligible_glob
0,linmap,0.62,0.444,0.357,-0.071,True,0.50,0.349,1.0,0.0,True
1,linmap_prompt,0.78,0.750,0.429,-0.143,True,0.95,0.884,1.0,0.0,True


**LoRA на глобальном срезе — строки подбора (val)**

,lr,epochs,adapter_id,path,train_seconds,n_examples,n_trainable,success,base_success,sensitivity,specificity,spec_drop,diff,ci_low,ci_high,fixed,broken
0,0.0,2,e56e9e74d6ff,/Users/admin/Desktop/steering-tool-feedback/re...,1060.4,112,3211264,0.726,0.339,1.0,0.0,1.0,0.387,0.065,0.677,41,17
1,0.0,2,d6261c8c0ddf,/Users/admin/Desktop/steering-tool-feedback/re...,1014.0,112,3211264,0.710,0.339,1.0,0.0,1.0,0.371,0.048,0.662,41,18


**seen, linmap_mix — разведка, вне H4′**

,method,success_cond,sens_cond,spec_cond,spec_drop_cond,eligible_cond,success_glob,sens_glob,spec_glob,spec_drop_glob,eligible_glob
0,linmap_mix,0.84,0.788,0.706,-0.412,True,0.52,0.171,1.0,0.0,True


**unseen, linmap_mix — разведка, вне H4′**

,method,success_cond,sens_cond,spec_cond,spec_drop_cond,eligible_cond,success_glob,sens_glob,spec_glob,spec_drop_glob,eligible_glob
0,linmap_mix,0.76,0.694,0.429,-0.143,True,0.42,0.302,1.0,0.0,True


## Вывод → журнал

In [6]:
display(Markdown(exp.report(
    conclusion="""РАЗВЕДКА (hypothesis=explore, test не открыт). Глобальный срез g1 (правило «после check_stock сначала спросить пользователя», в промпте не написано), Qwen3-1.7B, 160 задач (train 80 / val 31: 22 глобальных и 9 контрольных пар), отбор val+loto для обоих срезов (запись 2026-10-31 18:12), перенос seed 606. Прогон — повтор из кэша остановленного 20261031-181742_global-slice-qwen3-1.7b_0a9f9b (заметка 2026-11-01 21:23); val и перенос совпали с ним до третьего знака. Поправка к той заметке: конфигурации LoRA — lr 1e-4 и 3e-4, 2 эпохи, 112 примеров (а не r8/α, как написано в скобках).

1) H4′ по предрегистрации не решается: вердикт not_run. LoRA на g1 не выбрана — обе конфигурации выучили «всегда спрашивать»: принятие 1.0, сохранность 0.0 (падение 1.0 при допуске 0.05), успех на val 0.726 / 0.710 (исправила 41, сломала 17–18 веток). Поэтому на переносе LoRA не оценивалась, Δ_глоб и I не определены. Случай «ориентир сам не проходит допуск» в предрегистрации не описан; решения нет, оно за Павлом.

2) Описательно (успех / принятие / сохранность):
- val: база 0.34/0.00/1.00; policy_prompt 0.92/0.77/1.00; linmap_prompt 0.92/0.77/1.00 (L18/α2.0); linmap 0.48/0.14/1.00 (L16/α1.0); linmap_mix 0.47/0.14/1.00; CAA 0.39/0.05/1.00.
- перенос, знакомые типы: база 0.34/0.00/1.00; policy_prompt 0.99/0.97/1.00; linmap_prompt 0.98/0.94/1.00; linmap 0.52/0.20/1.00; linmap_mix 0.52/0.17/1.00; CAA 0.38/0.03/1.00.
- перенос, новые типы: база 0.41/0.26/1.00; policy_prompt 0.99/0.98/1.00; linmap_prompt 0.95/0.88/1.00; linmap 0.50/0.35/1.00; linmap_mix 0.42/0.30/1.00; CAA 0.48/0.30/1.00.
linmap_prompt почти повторяет своего учителя-промпт и не ломает сохранность — в отличие от LoRA на тех же данных.

3) Почему так (гипотеза, разведка). Обучающие примеры глобального среза — одно действие ask_user в обеих ветках; контрольных пар в обучении нет (как инвариантных в условном срезе). Из таких данных граница правила не видна, и LoRA выучила константу. linmap_prompt получает границу из учителя: правило словами («после проверки наличия сначала спросить») задаёт условие, и карта переносит его на контрольные состояния. Цель-центроид (linmap, linmap_mix) слаба: верных состояний базы на глобальных парах train всего 14 из 98; при большей силе она тоже начинает «всегда спрашивать» (в подборе L22/α1: принятие 0.95, сохранность 0.22).

4) Если сравнивать с условным срезом без вердикта: там LoRA впереди (0.94 против 0.73 у linmap_prompt на знакомых типах), здесь linmap_prompt (0.98) обходит негодную LoRA (val 0.73). Направление противоположно H4′ и совпадает со встречным предсказанием из FINDINGS F1, но держится на провале ориентира и на учителе, а не на сравнении способов вмешательства на одних данных. CAA на g1 слаб (0.38–0.48), так что «постоянный вектор хорош для глобального изменения» здесь не подтвердилось.

Ограничения: сохранность на val — 9 пар (шаг 0.11), на новых типах переноса — 7 контрольных пар (доли в наборе переноса не точные); у LoRA только две конфигурации; всё — разведка.""",
    next_step="""Решение Павла по H4′ (ориентир не прошёл допуск на глобальном срезе), варианты: (a) записать H4′ как «не решена по предрегистрации» и считать g1 неудачным срезом для сравнения с LoRA; (b) новая версия глобального среза, где граница правила видна в обучающих данных — контрольные пары в обучении всех обучаемых методов, одинаково в обоих срезах (меняет схему, фиксируется до прогона с датой); (c) разведочная строка «LoRA без допуска» как ориентир — вывод только разведочный. До решения новые прогоны по H4′ не запускать. Независимо от H4′: повторить linmap_mix (F8) на другом свежем наборе условного среза.""",
)))


## 2026-11-01 21:28 · прогон `20261101-212619_global-slice-qwen3-1.7b_0a9f9b`

**Гипотеза:** explore · **Ноутбук:** notebooks/12_global_slice.ipynb · **Модель:** Qwen/Qwen3-1.7B (mps, float16) · **Режим решения:** scored
**Стадии:** data, baseline, probe, vectors, sweep, fit, transfer · **git:** `cd3f0f9` (есть незакоммиченные изменения)

**Зачем:** H4′: глобальный срез g1 (правило ask_user) — linmap, linmap_prompt (учитель policy_prompt), LoRA; linmap_mix и CAA справочно.

**Отличия от конфига по умолчанию:** `act.alphas` = [1.0, 1.5, 2.0, 3.0, 4.0]; `act.ranks` = [8]; `act.select` = 'val+loto'; `act.teacher` = 'policy_prompt'; `data.generator` = 'g1'; `data.n_tasks` = 160; `data.stratify_pair_type` = True; `episode.enabled` = False; `eval.methods` = ['baseline', 'policy_prompt', 'steering', 'lora', 'linmap', 'linmap_prompt', 'linmap_mix']; `model.name` = 'Qwen/Qwen3-1.7B'; `transfer.episode_methods` = []; `transfer.seed` = 606

**⚠ Флаги протокола:**
- незакоммиченные изменения в src/docs/notebooks/configs — прогон не воспроизводится по коммиту
- изменены зафиксированные документы с первого прогона: HYPOTHESES.md, METRICS.md

**Результаты (val):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.339 | 0.000 | 1.000 | — |  | — | — | 798 | 10.000 | 391.900 |
| policy_prompt | 0.919 | 0.773 | 1.000 | 0.581 | [+0.419, +0.726] | 36.000 | 0.000 | 837 | 10.000 | 309.500 |
| steering | 0.387 | 0.045 | 1.000 | 0.048 | [+0.000, +0.113] | 3.000 | 0.000 | 798 | 10.000 | 408.200 |
| linmap | 0.484 | 0.136 | 1.000 | 0.145 | [+0.065, +0.242] | 9.000 | 0.000 | 798 | 10.000 | 459.700 |
| linmap_prompt | 0.919 | 0.773 | 1.000 | 0.581 | [+0.435, +0.726] | 36.000 | 0.000 | 798 | 10.000 | 301.900 |
| linmap_mix | 0.468 | 0.136 | 1.000 | 0.129 | [+0.048, +0.226] | 8.000 | 0.000 | 798 | 10.000 | 296.900 |
| linmap_prompt@val | 0.919 | 0.773 | 1.000 | 0.581 | [+0.419, +0.726] | 36.000 | 0.000 | 798 | 10.000 | 300.000 |

**Перенос, seen (разведка):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.340 | 0.000 | 1.000 | — |  | — | — | 800 | 10.000 | 306.400 |
| policy_prompt | 0.990 | 0.971 | 1.000 | 0.650 | [+0.520, +0.770] | 65.000 | 0.000 | 839 | 10.000 | 311.900 |
| steering | 0.380 | 0.029 | 1.000 | 0.040 | [+0.010, +0.080] | 4.000 | 0.000 | 800 | 10.000 | 311.800 |
| linmap | 0.520 | 0.200 | 1.000 | 0.180 | [+0.090, +0.270] | 18.000 | 0.000 | 800 | 10.000 | 312.200 |
| linmap_prompt | 0.980 | 0.943 | 1.000 | 0.640 | [+0.510, +0.760] | 64.000 | 0.000 | 800 | 10.000 | 312.600 |
| linmap_mix | 0.520 | 0.171 | 1.000 | 0.180 | [+0.100, +0.270] | 18.000 | 0.000 | 800 | 10.000 | 312.100 |
| linmap_prompt@val | 0.990 | 0.971 | 1.000 | 0.650 | [+0.520, +0.770] | 65.000 | 0.000 | 800 | 10.000 | 312.600 |

**Перенос, unseen (разведка):**

| method | success | sensitivity | specificity | diff_vs_base | ci | fixed | broken | tokens | calls | p50_ms |
|---|---|---|---|---|---|---|---|---|---|---|
| baseline | 0.410 | 0.256 | 1.000 | — |  | — | — | 796 | 10.000 | 309.000 |
| policy_prompt | 0.990 | 0.977 | 1.000 | 0.580 | [+0.450, +0.700] | 58.000 | 0.000 | 835 | 10.000 | 312.500 |
| steering | 0.480 | 0.302 | 1.000 | 0.070 | [+0.030, +0.120] | 7.000 | 0.000 | 796 | 10.000 | 312.700 |
| linmap | 0.500 | 0.349 | 1.000 | 0.090 | [+0.040, +0.150] | 9.000 | 0.000 | 796 | 10.000 | 313.000 |
| linmap_prompt | 0.950 | 0.884 | 1.000 | 0.540 | [+0.410, +0.660] | 54.000 | 0.000 | 796 | 10.000 | 312.400 |
| linmap_mix | 0.420 | 0.302 | 1.000 | 0.010 | [-0.040, +0.060] | 4.000 | 3.000 | 796 | 10.000 | 311.800 |
| linmap_prompt@val | 0.960 | 0.930 | 0.857 | 0.550 | [+0.420, +0.680] | 56.000 | 1.000 | 796 | 10.000 | 312.400 |

**Вердикты (автоматически, по правилам HYPOTHESES.md):**
- H1: ✔ support — лучший probe L1/field_read: AUROC 0.873, acc 0.855 при перестановке 0.372
- H2: ✔ support — различает (H1), но ошибается в 100% меняющих пар: чувствительность 0.000, специфичность 1.000
- H3: ? inconclusive (val) — на val: успех 0.339 → 0.387 при L16/post_message/α=0.25; решает только test
- H4: · not_run — нужны steering и LoRA, выбранные на val и оценённые на одних задачах
- ALT: · not_run — steering не сравнивался с промптом / пересказом / явным состоянием

**Вывод:** РАЗВЕДКА (hypothesis=explore, test не открыт). Глобальный срез g1 (правило «после check_stock сначала спросить пользователя», в промпте не написано), Qwen3-1.7B, 160 задач (train 80 / val 31: 22 глобальных и 9 контрольных пар), отбор val+loto для обоих срезов (запись 2026-10-31 18:12), перенос seed 606. Прогон — повтор из кэша остановленного 20261031-181742_global-slice-qwen3-1.7b_0a9f9b (заметка 2026-11-01 21:23); val и перенос совпали с ним до третьего знака. Поправка к той заметке: конфигурации LoRA — lr 1e-4 и 3e-4, 2 эпохи, 112 примеров (а не r8/α, как написано в скобках).

1) H4′ по предрегистрации не решается: вердикт not_run. LoRA на g1 не выбрана — обе конфигурации выучили «всегда спрашивать»: принятие 1.0, сохранность 0.0 (падение 1.0 при допуске 0.05), успех на val 0.726 / 0.710 (исправила 41, сломала 17–18 веток). Поэтому на переносе LoRA не оценивалась, Δ_глоб и I не определены. Случай «ориентир сам не проходит допуск» в предрегистрации не описан; решения нет, оно за Павлом.

2) Описательно (успех / принятие / сохранность):
- val: база 0.34/0.00/1.00; policy_prompt 0.92/0.77/1.00; linmap_prompt 0.92/0.77/1.00 (L18/α2.0); linmap 0.48/0.14/1.00 (L16/α1.0); linmap_mix 0.47/0.14/1.00; CAA 0.39/0.05/1.00.
- перенос, знакомые типы: база 0.34/0.00/1.00; policy_prompt 0.99/0.97/1.00; linmap_prompt 0.98/0.94/1.00; linmap 0.52/0.20/1.00; linmap_mix 0.52/0.17/1.00; CAA 0.38/0.03/1.00.
- перенос, новые типы: база 0.41/0.26/1.00; policy_prompt 0.99/0.98/1.00; linmap_prompt 0.95/0.88/1.00; linmap 0.50/0.35/1.00; linmap_mix 0.42/0.30/1.00; CAA 0.48/0.30/1.00.
linmap_prompt почти повторяет своего учителя-промпт и не ломает сохранность — в отличие от LoRA на тех же данных.

3) Почему так (гипотеза, разведка). Обучающие примеры глобального среза — одно действие ask_user в обеих ветках; контрольных пар в обучении нет (как инвариантных в условном срезе). Из таких данных граница правила не видна, и LoRA выучила константу. linmap_prompt получает границу из учителя: правило словами («после проверки наличия сначала спросить») задаёт условие, и карта переносит его на контрольные состояния. Цель-центроид (linmap, linmap_mix) слаба: верных состояний базы на глобальных парах train всего 14 из 98; при большей силе она тоже начинает «всегда спрашивать» (в подборе L22/α1: принятие 0.95, сохранность 0.22).

4) Если сравнивать с условным срезом без вердикта: там LoRA впереди (0.94 против 0.73 у linmap_prompt на знакомых типах), здесь linmap_prompt (0.98) обходит негодную LoRA (val 0.73). Направление противоположно H4′ и совпадает со встречным предсказанием из FINDINGS F1, но держится на провале ориентира и на учителе, а не на сравнении способов вмешательства на одних данных. CAA на g1 слаб (0.38–0.48), так что «постоянный вектор хорош для глобального изменения» здесь не подтвердилось.

Ограничения: сохранность на val — 9 пар (шаг 0.11), на новых типах переноса — 7 контрольных пар (доли в наборе переноса не точные); у LoRA только две конфигурации; всё — разведка.

**Следующий шаг:** Решение Павла по H4′ (ориентир не прошёл допуск на глобальном срезе), варианты: (a) записать H4′ как «не решена по предрегистрации» и считать g1 неудачным срезом для сравнения с LoRA; (b) новая версия глобального среза, где граница правила видна в обучающих данных — контрольные пары в обучении всех обучаемых методов, одинаково в обоих срезах (меняет схему, фиксируется до прогона с датой); (c) разведочная строка «LoRA без допуска» как ориентир — вывод только разведочный. До решения новые прогоны по H4′ не запускать. Независимо от H4′: повторить linmap_mix (F8) на другом свежем наборе условного среза.
